# Import

In [2]:
import numpy as np
import json
from scipy.sparse import load_npz,save_npz,diags,csr_matrix
import scipy.sparse as sp
import pandas as pd
import os
import requests
from io import BytesIO
from tqdm import tqdm
from scipy.sparse.linalg import eigsh
from scipy.spatial.distance import pdist, squareform
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.backends.backend_pdf import PdfPages
from pypdf import PdfReader, PdfWriter
from tempfile import NamedTemporaryFile
import networkx as nx
import pickle
import gseapy as gp
import mygene
from IPython.display import display, HTML
import re
from collections import deque
from goatools.obo_parser import GODag
import math
from itertools import combinations
from collections import Counter
from gseapy.parser import read_gmt
import time
import random
import ast

In [3]:
pd.set_option('display.width', None)      # No line-wrapping
pd.set_option('display.max_columns', None)  # Show all columns

# Prep

## Loading variables

In [4]:
DISEASE = input("Disease: ")
DISEASE_FOLDER = f"../output/{DISEASE}/"
RESULT_FOLDER = DISEASE_FOLDER + "leiden_results"
DGIDB_DIRECTORY = f"../../Gen_Hypergraph/output/DGIDB_{DISEASE}/"
MSIGDB_DIRECTORY = "../../Gen_Hypergraph/output/MSigDB_Full/"
RESULT_GRAPH = "result_graph"

with open(DISEASE_FOLDER + "gene_to_index_distinct.json", "r") as file:
    gene_to_index_distinct = json.load(file)
    
try:
    with open(DGIDB_DIRECTORY + f"gene_to_index.json", "r") as file:
        DGIDB_gene_to_index = json.load(file)
except FileNotFoundError:
    DGIDB_gene_to_index = {}
    print("File not found. Setting DGIDB_gene_to_index to be {}.")

In [5]:
## ORIGINAL
index_to_gene_distinct = {v: k for k, v in gene_to_index_distinct.items()}

In [6]:
# Loading result graph and communities
with open(f"{RESULT_FOLDER}/result_communities_selected.pkl", "rb") as f:
    communities_selected = pickle.load(f)
with open(f"{RESULT_FOLDER}/result_communities.pkl", "rb") as f:
    communities = pickle.load(f)
with open(f"{RESULT_FOLDER}/{RESULT_GRAPH}.pkl", "rb") as f:
    graph = pickle.load(f)

In [7]:
for c in communities:
    print(len(c))

4072
3529
3296
3195
2633
2476
1183
773
434
43
26
20
15
11
11
8
8
7
7
5
5
4
4
3
2
2
2
2
2
2
2
2
2
2
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1
1


## Helpful functions (big object, drop NAN)

In [8]:
# Helpful functions
def drop_nan_from_communities(communities):
    cleaned_communities = []
    total_dropped = 0

    for i, community in enumerate(communities):
        cleaned = []
        dropped = 0
        for g in community:
            if g is None or (isinstance(g, float) and math.isnan(g)):
                dropped += 1
            else:
                cleaned.append(g)
        cleaned_communities.append(cleaned)
        total_dropped += dropped
        print(f"Community {i}: dropped {dropped} NaN entries")

    print(f"\nTotal dropped across all communities: {total_dropped}")
    return cleaned_communities

def big_objects(n=10, min_mb=1):
    """
    Show the largest objects currently in memory.
    
    Parameters
    ----------
    n : int
        Number of top objects to show.
    min_mb : float
        Minimum size (in MB) to include.
    """
    import sys
    import numpy as np
    import pandas as pd
    import scipy.sparse as sp
    from IPython import get_ipython

    def get_size(obj):
        try:
            if isinstance(obj, np.ndarray):
                return obj.nbytes
            elif isinstance(obj, pd.DataFrame) or isinstance(obj, pd.Series):
                return obj.memory_usage(deep=True).sum()
            elif sp.issparse(obj):
                return (obj.data.nbytes +
                        obj.indptr.nbytes +
                        obj.indices.nbytes)
            else:
                return sys.getsizeof(obj)
        except Exception:
            return 0

    ip = get_ipython()
    if ip is None:
        ns = globals()
    else:
        ns = ip.user_ns

    items = []
    for name, val in ns.items():
        if name.startswith('_'):
            continue  # skip internals
        size = get_size(val)
        if size > min_mb * 1024 ** 2:
            items.append((name, type(val).__name__, size))

    items.sort(key=lambda x: x[2], reverse=True)

    print(f"{'Variable':30s} {'Type':25s} {'Size (MB)':>10s}")
    print("-" * 70)
    for name, t, size in items[:n]:
        print(f"{name:30s} {t:25s} {size / 1024 ** 2:10.2f}")

## Index to NCBI

In [9]:
# Convert index to ncbi
def index_to_ncbi(comms,index_to_ncbi_dict = index_to_gene_distinct):
    comms_ncbi = [list(map(index_to_ncbi_dict.get, c)) for c in comms]
    return comms_ncbi

In [10]:
communities_ncbi = index_to_ncbi(communities_selected,index_to_gene_distinct)
print(communities_ncbi)
print(len(communities_ncbi))
with open(f"{RESULT_FOLDER}/result_communities_ncbi_selected.pkl", "wb") as f:
    pickle.dump(communities_ncbi, f)

[['10043', '50810', '83941', '59342', '285527', '26064', '64744', '55852', '9922', '9240', '54972', '51389', '387263', '64943', '223082', '473', '9764', '55137', '222658', '23164', '163259', '30846', '51108', '79144', '11275', '4299', '25966', '122416', '4188', '23301', '1039', '51375', '83543', '23589', '29994', '55075', '10489', '55556', '1266', '84288', '4076', '115123', '64222', '148867', '4302', '57648', '140890', '6238', '57458', '25941', '1318', '51015', '5128', '84909', '23255', '10534', '6856', '25801', '29966', '56889', '5098', '84549', '55905', '5523', '79629', '84365', '9703', '57561', '4814', '65244', '55848', '25864', '64343', '56061', '8804', '23484', '7572', '25875', '55012', '10330', '9920', '10217', '79090', '55608', '7257', '203197', '51303', '119032', '83641', '26750', '9399', '8522', '9338', '56172', '80263', '8568', '51088', '9857', '23200', '55196', '64778', '9887', '5547', '9191', '79022', '9601', '27333', '23008', '26017', '51678', '10129', '56256', '57460', '5

In [11]:
communities_ncbi_full = index_to_ncbi(communities,index_to_gene_distinct)
print(communities_ncbi_full)
print(len(communities_ncbi_full))
with open(f"{RESULT_FOLDER}/result_communities_ncbi.pkl", "wb") as f:
    pickle.dump(communities_ncbi_full, f)

[['9', '16', '22', '28', '31', '52', '53', '54', '59', '72', '95', '97', '98', '120', '123', '132', '162', '165', '174', '176', '202', '248', '249', '250', '259', '270', '271', '272', '280', '293', '308', '318', '319', '323', '332', '334', '339', '343', '354', '372', '377', '379', '381', '390', '393', '394', '398', '410', '412', '414', '415', '416', '417', '419', '421', '432', '443', '462', '466', '473', '475', '516', '517', '518', '526', '528', '529', '533', '534', '547', '590', '597', '599', '605', '607', '608', '629', '632', '633', '639', '642', '656', '663', '664', '670', '676', '685', '686', '689', '701', '705', '712', '713', '716', '720', '726', '729', '730', '732', '733', '734', '740', '744', '745', '757', '759', '761', '763', '767', '768', '771', '793', '800', '813', '820', '821', '822', '826', '827', '828', '830', '833', '862', '866', '868', '871', '892', '900', '901', '913', '915', '919', '924', '925', '932', '936', '938', '942', '951', '956', '962', '963', '968', '969', '971

## NCBI to HGNC

In [12]:
hgnc = pd.read_csv("../../Data/hgnc_complete_set.txt", sep="\t", dtype=str)
ncbi_to_hgnc_dict = dict(
    zip(
        hgnc["entrez_id"].dropna(),
        hgnc.loc[hgnc["entrez_id"].notna(), "symbol"]
    )
)

def ncbi_to_HGNC(comms_ncbi):
    comms_HGNC = []
    for community in comms_ncbi:
        symbols = [ncbi_to_hgnc_dict.get(n) for n in community]
        comms_HGNC.append(symbols)
    return comms_HGNC

In [13]:
# # NCBI to HGNC symbol
# def ncbi_to_HGNC(comms_ncbi):
#     comms_HGNC = []
#     for community in comms_ncbi:
#         mg = mygene.MyGeneInfo()
#         entrez_ids = [str(e) for e in community]

#         results = mg.querymany(
#             entrez_ids,
#             scopes="entrezgene",
#             fields="symbol",
#             species="human"
#         )

#         # Build a mapping: input ID -> symbol (or None)
#         id_to_symbol = {}
#         for r in results:
#             q = str(r.get("query"))
#             id_to_symbol[q] = r.get("symbol") if not r.get("notfound") else None

#         # Preserve original order
#         symbols = [id_to_symbol.get(str(e), None) for e in entrez_ids]
#         comms_HGNC.append(symbols)
#     return comms_HGNC


In [14]:
COMMUNITIES_HGNC = ncbi_to_HGNC(communities_ncbi)
COMMUNITIES_HGNC_full = ncbi_to_HGNC(communities_ncbi_full)

In [15]:
print(len(COMMUNITIES_HGNC))

10


In [16]:
COMMUNITIES_HGNC = drop_nan_from_communities(COMMUNITIES_HGNC)
COMMUNITIES_HGNC_full = drop_nan_from_communities(COMMUNITIES_HGNC_full)

Community 0: dropped 0 NaN entries
Community 1: dropped 5 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 11 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 0 NaN entries
Community 7: dropped 0 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 0 NaN entries

Total dropped across all communities: 16
Community 0: dropped 0 NaN entries
Community 1: dropped 8 NaN entries
Community 2: dropped 0 NaN entries
Community 3: dropped 28 NaN entries
Community 4: dropped 0 NaN entries
Community 5: dropped 0 NaN entries
Community 6: dropped 1 NaN entries
Community 7: dropped 1 NaN entries
Community 8: dropped 0 NaN entries
Community 9: dropped 0 NaN entries
Community 10: dropped 0 NaN entries
Community 11: dropped 0 NaN entries
Community 12: dropped 0 NaN entries
Community 13: dropped 0 NaN entries
Community 14: dropped 0 NaN entries
Community 15: dropped 0 NaN entries
Community 16: dropped 0 NaN entries
Comm

In [17]:
with open(f"{RESULT_FOLDER}/result_communities_HGNC_selected.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC, f)
with open(f"{RESULT_FOLDER}/result_communities_HGNC.pkl", "wb") as f:
    pickle.dump(COMMUNITIES_HGNC_full, f)

In [18]:
print(len(COMMUNITIES_HGNC))
print(len(COMMUNITIES_HGNC_full))

10
237


# Categoization Prep

### GO-slim

In [19]:
DATA_DIRECTORY = "../../data"
GO_OBO = f"{DATA_DIRECTORY}/GO/go-basic.obo"            # put the file in your working dir (or give full path)
GOSLIM_OBO = f"{DATA_DIRECTORY}/GO/goslim_generic.obo"  # swap to another slim if you prefer
GOSLIM_PIR_OBO = f"{DATA_DIRECTORY}/GO/goslim_pir.obo"  # swap to another slim if you prefer
GOSLIM_YEAST_OBO = f"{DATA_DIRECTORY}/GO/goslim_yeast.obo"
GOSLIM_AGR_OBO = f"{DATA_DIRECTORY}/GO/goslim_agr.obo"

In [20]:
# GO library
go = GODag(GO_OBO)

# SLIM libraries
slim = GODag(GOSLIM_OBO)
slim_pir = GODag(GOSLIM_PIR_OBO)
slim_yeast = GODag(GOSLIM_YEAST_OBO)
slim_agr = GODag(GOSLIM_AGR_OBO)

slim_ids = set(slim.keys())
slim_pir_ids = set(slim_pir.keys())
slim_yeast_ids = set(slim_yeast.keys())
slim_agr_ids = set(slim_agr.keys())

../../data/GO/go-basic.obo: fmt(1.2) rel(2025-10-10) 42,666 Terms
../../data/GO/goslim_generic.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_generic.owl) 205 Terms
../../data/GO/goslim_pir.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_pir.owl) 617 Terms
../../data/GO/goslim_yeast.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_yeast.owl) 295 Terms
../../data/GO/goslim_agr.obo: fmt(1.2) rel(go/2025-10-10/subsets/goslim_agr.owl) 94 Terms


In [21]:
GO_RE = re.compile(r"(GO:\d{7})")

def get_goid(term: str):
    if isinstance(term, str):
        m = GO_RE.search(term)
        if m:
            return m.group(1)
    raise RuntimeError("Term not found!!")

def get_go_ancestors(go_id):
    """Return a list of ancestor GO term IDs for the given GO ID using QuickGO."""
    url = f"https://www.ebi.ac.uk/QuickGO/services/ontology/go/terms/{go_id}/ancestors"
    headers = {"Accept": "application/json"}

    r = requests.get(url, headers=headers)
    r.raise_for_status()

    data = r.json()
    results = data.get("results", [])
    if not results:
        return []

    # Ancestors come back as a simple list of GO IDs (strings)
    ancestors = results[0].get("ancestors", [])
    return set(ancestors)


def get_go_ancestors_in_slim(go_id):
    ancestors = get_go_ancestors(go_id)
    return slim_ids & ancestors

In [22]:
def get_go_ancestors_at_depth(go_id, depth, include_relations=("is_a", "part_of")):
    """
    Return the set of GO term IDs that are ancestors of `go_id` and have
    absolute depth == `depth` in the GO DAG.

    Parameters
    ----------
    go_id : str
        Starting GO term (e.g., "GO:0051310").
    depth : int
        Absolute depth in the GO DAG (e.g., 3 means all ancestors at depth=3).
    include_relations : tuple[str]
        Relation types to traverse upward, e.g. ("is_a", "part_of", "regulates", ...).

    Returns
    -------
    set[str]
        Ancestor GO IDs whose term.depth == `depth`. Empty set if none.
    """
    if depth < 0:
        return set()
    if go_id not in go:
        return set()

    # One-hop function honoring relation filter
    def parent_ids(term):
        ids = set()
        if "is_a" in include_relations:
            # GOATOOLS usually puts is_a parents here (and sometimes part_of merged)
            ids.update(p.id for p in term.parents)

        rel = getattr(term, "relationship", {}) or {}
        for r in include_relations:
            # relationship entries are already GO IDs
            ids.update(rel.get(r, []))

        # ensure IDs exist in DAG
        return {pid for pid in ids if pid in go}

    result = set()
    frontier = {go_id}
    visited = {go_id}

    # BFS upwards, but pruning branches that are already above the target depth
    while frontier:
        next_frontier = set()
        for node in frontier:
            for pid in parent_ids(go[node]):
                if pid in visited:
                    continue
                visited.add(pid)
                d = go[pid].depth  # absolute depth in DAG

                if d == depth:
                    # ancestor at the exact target depth
                    result.add(pid)
                elif d > depth:
                    # still "below" target depth (further from root),
                    # its parents might reach the target depth
                    next_frontier.add(pid)
                # if d < depth: this branch has gone above the target,
                # and all further ancestors will have depth <= d, so we can skip
        frontier = next_frontier

    return result


### KEGG

In [23]:
def build_kegg_name_to_id(species="hsa"):
    """Map KEGG pathway name -> 'hsaXXXXX' (species-specific)."""
    lines = requests.get(f"https://rest.kegg.jp/list/pathway/{species}").text.strip().splitlines()
    name_to_id = {}
    for ln in lines:
        pid, raw = ln.split("\t")
        pid = pid.replace("path:", "")  # e.g. hsa03010
        # strip " - Homo sapiens (human)" suffix
        name = re.sub(r"\s*-\s*Homo sapiens.*$", "", raw).strip()
        name_to_id[name.lower()] = pid
    return name_to_id

name_to_id = build_kegg_name_to_id("hsa")

In [24]:
def get_kegg_level2(hsa_id: str) -> str | None:
    """
    Return the KEGG Level 2 category for a pathway like 'hsa03040'.
    Example: get_kegg_level2("hsa03040") -> 'Transcription'
    """
    url = f"http://rest.kegg.jp/get/{hsa_id}"
    try:
        text = requests.get(url, timeout=10).text
    except Exception:
        return None

    for line in text.splitlines():
        if line.startswith("CLASS"):
            # CLASS line looks like: CLASS       Genetic Information Processing; Transcription
            parts = [p.strip() for p in line.split(";", maxsplit=2)]
            if len(parts) >= 2:
                return [parts[1]]
            elif len(parts) == 1:
                return [parts[0].replace("CLASS", "").strip()]
    return []

### Reactome

In [25]:
def build_reactome_level_map(level=1, species="9606"):
    """
    Returns { 'R-HSA-xxxxx': ['CategoryNameAtLevel', ...], ... } for the given species.

    Parameters
    ----------
    level : int, default=1
        1-based depth in the Reactome pathway hierarchy:
          - level=1 → top-level Reactome categories (original behavior)
          - level=2 → second-level ancestors, etc.
        If a node is shallower than `level`, the deepest available ancestor
        is used as a fallback.
    species : str, default="9606"
        Taxonomy ID ("9606") or species name ("Homo sapiens").
    """
    if level < 1:
        raise ValueError("level must be >= 1 (1-based depth)")

    # ensure spaces are encoded if a name is used
    species_path = species.replace(" ", "+")
    url = f"https://reactome.org/ContentService/data/eventsHierarchy/{species_path}"
    print(url)
    r = requests.get(url, headers={"Accept": "application/json"}, timeout=300)
    r.raise_for_status()
    trees = r.json()  # list of trees, one per TopLevelPathway

    mapping = {}

    def walk(node, ancestors):
        """
        node: current node dict
        ancestors: list of ancestor nodes from root to parent of `node`
        """
        # ancestors_chain includes current node at the end
        ancestors_chain = ancestors + [node]

        st_id = node.get("stId")
        if st_id:
            # We want the ancestor at depth `level` (1-based).
            # If the path is shorter than `level`, fall back to the deepest one.
            if len(ancestors_chain) >= level:
                cat_node = ancestors_chain[level - 1]
            else:
                cat_node = ancestors_chain[-1]

            cat_name = cat_node.get("name")
            if cat_name:
                mapping.setdefault(st_id, set()).add(cat_name)

        # Recurse into children
        for child in node.get("children", []):
            walk(child, ancestors_chain)

    # Each tree is a top-level pathway
    for top in trees:
        walk(top, [])

    # sets -> sorted lists
    return {k: sorted(v) for k, v in mapping.items()}

In [26]:
# Specific for Reactome: build level map first
reactome_level1 = build_reactome_level_map(level = 1)

https://reactome.org/ContentService/data/eventsHierarchy/9606


# Run Enrichment Analysis

In [27]:
TERM_SCORE_CAP = 0.001
PERCENTAGE = 0.1

### GO

In [28]:
# GO Analysis; save terms with small size and high p-value
def go_enrichment(communities,
                  term_score_cap,
                  percentage, 
                  slim_ids = slim_yeast_ids,
                  depth = 1):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    
    for community in communities:
        # Gene Ontology enrichment
        enr_go = gp.enrichr(
            gene_list=community,
            gene_sets=['GO_Biological_Process_2023',
                    'GO_Molecular_Function_2023',
                    'GO_Cellular_Component_2023'],
            organism='Human',
            outdir=None # don't write to disk
        )
        go_df = enr_go.results
        

        # Filter by overlap percentage and adjusted p-value
        mask =  (go_df["Adjusted P-value"] < term_score_cap) & (go_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = go_df[mask].copy()
        
        # Categorization from GO-Slim
        filtered["GO_ID"] = filtered["Term"].apply(get_goid)
        # filtered["Slim_IDs"] = filtered["GO_ID"].apply(get_go_ancestors_in_slim)
        filtered["Slim_IDs"] = filtered["GO_ID"].apply(lambda id: get_go_ancestors_at_depth(id, depth=depth, include_relations=("is_a", "part_of")))
        
        # Get empty count
        empty_count = (filtered["Slim_IDs"].apply(len) == 0).sum()
        
        # Get slim names    
        filtered["Category"] = filtered["Slim_IDs"].apply(lambda ids: [go[i].name for i in ids])
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            print(f"Number of unmapped terms: {empty_count}")      
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Slim_IDs","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [29]:
go_important_terms = go_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE,slim_ids,depth = 1)

Size of community: 1536
Number of filtered terms: 8
Number of unmapped terms: 0


C:\Users\celem\AppData\Local\Temp\ipykernel_98744\3881098508.py:53: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
2494,0,COPI Vesicle Coat (GO:0030126),7/12,6.302583e-04,{GO:0032991},[protein-containing complex]
2491,0,Small-Subunit Processome (GO:0032040),22/73,2.870727e-06,{GO:0032991},[protein-containing complex]
2,0,Ribosomal Small Subunit Biogenesis (GO:0042274),21/84,6.811090e-04,{GO:0009987},[cellular process]
1,0,rRNA Processing (GO:0006364),24/101,4.860535e-04,{GO:0009987},[cellular process]
0,0,Ribosome Biogenesis (GO:0042254),32/155,4.369339e-04,{GO:0009987},[cellular process]
2017,0,RNA Binding (GO:0003723),205/1411,3.059371e-17,{GO:0005488},[binding]
2492,0,Nucleolus (GO:0005730),103/771,2.870727e-06,{GO:0110165},[cellular anatomical structure]
2493,0,Nuclear Lumen (GO:0031981),103/780,3.483481e-06,{GO:0110165},[cellular anatomical structure]


Size of community: 1293
Number of filtered terms: 63
Number of unmapped terms: 4


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
4121,2,THO Complex Part Of Transcription Export Complex (GO:0000445),5/5,0.000033,{GO:0032991},[protein-containing complex]
33,2,Positive Regulation Of rRNA Processing (GO:2000234),6/9,0.000520,{GO:0065007},[biological regulation]
16,2,Formation Of Cytoplasmic Translation Initiation Complex (GO:0001732),8/14,0.000128,{GO:0009987},[cellular process]
22,2,mRNA Methylation (GO:0080009),8/15,0.000192,{},[]
3459,2,Lysophosphatidic Acid Acyltransferase Activity (GO:0042171),10/20,0.000014,{GO:0003824},[catalytic activity]
3461,2,1-Acylglycerol-3-Phosphate O-acyltransferase Activity (GO:0003841),9/19,0.000076,{GO:0003824},[catalytic activity]
4126,2,COPII Vesicle Coat (GO:0030127),7/15,0.000374,{GO:0032991},[protein-containing complex]
3465,2,UDP-galactosyltransferase Activity (GO:0035250),8/19,0.000650,{GO:0003824},[catalytic activity]
32,2,Negative Regulation Of Macroautophagy (GO:0016242),9/22,0.000463,{GO:0065007},[biological regulation]
13,2,Phospholipid Biosynthetic Process (GO:0008654),11/27,0.000097,{GO:0009987},[cellular process]


...

,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
849,6,Phosphatidate Phosphatase Activity (GO:0008195),5/13,1.890431e-04,{GO:0003824},[catalytic activity]
848,6,Lipid Phosphatase Activity (GO:0042577),5/13,1.890431e-04,{GO:0003824},[catalytic activity]
850,6,G Protein-Coupled Photoreceptor Activity (GO:0008020),5/14,2.722722e-04,{GO:0060089},[molecular transducer activity]
11,6,Dopaminergic Neuron Differentiation (GO:0071542),6/19,5.956453e-04,"{GO:0032502, GO:0009987}","[developmental process, cellular process]"
844,6,Neuropeptide Hormone Activity (GO:0005184),8/26,5.010853e-06,"{GO:0005488, GO:0098772}","[binding, molecular function regulator activity]"
843,6,Neuropeptide Activity (GO:0160041),8/26,5.010853e-06,"{GO:0005488, GO:0098772}","[binding, molecular function regulator activity]"
841,6,Neuropeptide Receptor Activity (GO:0008188),11/36,4.162322e-08,{GO:0060089},[molecular transducer activity]
3,6,Neuropeptide Signaling Pathway (GO:0007218),19/68,3.483922e-12,"{GO:0065007, GO:0009987}","[biological regulation, cellular process]"
840,6,G Protein-Coupled Peptide Receptor Activity (GO:0008528),20/77,3.584127e-13,{GO:0060089},[molecular transducer activity]
10,6,Positive Regulation Of Cytosolic Calcium Ion Concentration Involved In Phospholipase C-activating G Protein-Coupled Signaling Pathway (GO:0051482),7/27,4.879618e-04,{},[]


Size of community: 275
Number of filtered terms: 5
Number of unmapped terms: 0


,Community Index,Term,Overlap,Adjusted P-value,Slim_IDs,Category
2,8,Detection Of Chemical Stimulus Involved In Sensory Perception Of Smell (GO:0050911),86/139,9.420555e-128,{GO:0050896},[response to stimulus]
1,8,Detection Of Chemical Stimulus Involved In Sensory Perception (GO:0050907),87/141,5.591666e-129,{GO:0050896},[response to stimulus]
17,8,Olfactory Receptor Activity (GO:0004984),220/362,0.000000e+00,{GO:0060089},[molecular transducer activity]
0,8,Sensory Perception Of Smell (GO:0007608),137/230,1.431229e-206,{GO:0032501},[multicellular organismal process]
3,8,Sensory Perception Of Chemical Stimulus (GO:0007606),61/110,2.837368e-85,{GO:0032501},[multicellular organismal process]


6 out of 10 communities had significant GO terms.


c:\Users\celem\AppData\Local\Programs\Python\Python310\lib\site-packages\gseapy\enrichr.py:689: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  self.results = pd.concat(self.results, ignore_index=True)


In [30]:
go_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value)
0,0,1536,COPI Vesicle Coat (GO:0030126),7/12,6.302583e-04,[protein-containing complex],GO_Cellular_Component_2023,8.723299e-06,0.0,0.0,16.901635,1.968958e+02,COPA;COPB1;TMED3;COPZ2;COPG1;TMED7;COPE,GO:0030126,{GO:0032991},0.583333
1,0,1536,Small-Subunit Processome (GO:0032040),22/73,2.870727e-06,[protein-containing complex],GO_Cellular_Component_2023,1.460568e-08,0.0,0.0,5.246277,9.465256e+01,NOP56;WDR36;NOP14;KRR1;PNO1;UTP3;WDR3;MRPS12;F...,GO:0032040,{GO:0032991},0.301370
2,0,1536,Ribosomal Small Subunit Biogenesis (GO:0042274),21/84,6.811090e-04,[cellular process],GO_Biological_Process_2023,1.013053e-06,0.0,0.0,4.048625,5.588132e+01,NOP56;WDR36;NOP14;KRR1;PNO1;UTP3;WDR3;MRPS12;F...,GO:0042274,{GO:0009987},0.250000
3,0,1536,rRNA Processing (GO:0006364),24/101,4.860535e-04,[cellular process],GO_Biological_Process_2023,4.819568e-07,0.0,0.0,3.790353,5.513224e+01,NOP56;NOP14;WDR36;DDX27;WDR3;MAK16;DDX10;DDX52...,GO:0006364,{GO:0009987},0.237624
4,0,1536,Ribosome Biogenesis (GO:0042254),32/155,4.369339e-04,[cellular process],GO_Biological_Process_2023,2.166256e-07,0.0,0.0,3.172634,4.868438e+01,DDX27;WDR3;NIP7;MRPS12;FCF1;WDR46;MRM2;NOL6;RR...,GO:0042254,{GO:0009987},0.206452
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
863,8,275,Detection Of Chemical Stimulus Involved In Sen...,86/139,9.420555e-128,[response to stimulus],GO_Biological_Process_2023,1.662451e-128,0.0,0.0,168.892083,4.969187e+04,OR10J1;OR2A1;OR4E2;OR10J3;OR4E1;OR10J5;OR10J4;...,GO:0050911,{GO:0050896},0.618705
864,8,275,Detection Of Chemical Stimulus Involved In Sen...,87/141,5.591666e-129,[response to stimulus],GO_Biological_Process_2023,6.578431e-130,0.0,0.0,168.575355,5.014312e+04,OR10J1;OR2A1;OR4E2;OR10J3;OR4E1;OR10J5;OR10J4;...,GO:0050907,{GO:0050896},0.617021
865,8,275,Olfactory Receptor Activity (GO:0004984),220/362,0.000000e+00,[molecular transducer activity],GO_Molecular_Function_2023,0.000000e+00,0.0,0.0,551.633803,inf,OR7G2;OR52N1;OR2M5;OR2M4;OR10AC1;OR51L1;OR2AE1...,GO:0004984,{GO:0060089},0.607735
866,8,275,Sensory Perception Of Smell (GO:0007608),137/230,1.431229e-206,[multicellular organismal process],GO_Biological_Process_2023,8.418996e-208,0.0,0.0,209.567087,9.992310e+04,OR1C1;OR2M5;OR2M4;OR2T12;OR2T10;OR10AC1;OR2T11...,GO:0007608,{GO:0032501},0.595652


### KEGG

In [31]:
# KEGG
def kegg_enrichment(communities,
                    term_score_cap,
                    percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['KEGG_2021_Human'],
            organism='Human',
            outdir=None
        )
        KEGG_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = KEGG_df[mask].copy()
        
        # Categorization from KEGG Level 2
        filtered["KEGG_ID"] = filtered["Term"].str.replace(r"\s*-\s*Homo sapiens.*$", "", regex=True).str.lower().map(name_to_id)
        filtered["Category"] = filtered["KEGG_ID"].map(get_kegg_level2)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            # print size of community
            print(f"Size of community: {len(community)}")   
            
            # print number of filtered terms
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            
            # show results
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"KEGG_ID","Category"]].head(10).to_html(max_cols=None)))
            num_nonzero_communities += 1

        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [32]:
kegg_important_terms = kegg_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

Size of community: 1207
Number of filtered terms: 1


C:\Users\celem\AppData\Local\Temp\ipykernel_98744\1053027199.py:38: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,1,Herpes simplex virus 1 infection,61/498,0.000006,hsa05168,[Infectious disease: viral]


Size of community: 1293
Number of filtered terms: 6


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
2,2,Mucin type O-glycan biosynthesis,13/36,1.341216e-05,hsa00512,[Glycan biosynthesis and metabolism]
1,2,N-Glycan biosynthesis,17/50,7.557622e-07,hsa00510,[Glycan biosynthesis and metabolism]
3,2,Sphingolipid metabolism,15/49,1.411786e-05,hsa00600,[Lipid metabolism]
4,2,Glycosphingolipid biosynthesis,13/45,1.471350e-04,NaN,[]
5,2,Glycosaminoglycan biosynthesis,13/53,8.703755e-04,NaN,[]
0,2,RNA transport,37/186,1.563500e-07,NaN,[]


...

Size of community: 552
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,6,Neuroactive ligand-receptor interaction,71/341,8.350459e-40,hsa04080,[Signaling molecules and interaction]


Size of community: 275
Number of filtered terms: 1


,Community Index,Term,Overlap,Adjusted P-value,KEGG_ID,Category
0,8,Olfactory transduction,266/440,0.0,hsa04740,[Sensory system]


5 out of 10 communities had significant GO terms.


In [33]:
kegg_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,KEGG_ID,Overlap (value)
0,1,1207,Herpes simplex virus 1 infection,61/498,5.765486e-06,[Infectious disease: viral],KEGG_2021_Human,1.087827e-07,0.0,0.0,2.235846,35.849356,ZNF133;ZNF573;ZNF253;ZNF250;ZFP82;ZNF83;ZSCAN3...,hsa05168,0.122490
1,2,1293,Mucin type O-glycan biosynthesis,13/36,1.341216e-05,[Glycan biosynthesis and metabolism],KEGG_2021_Human,1.845710e-07,0.0,0.0,8.250408,127.924480,GALNT12;GALNT7;GALNT11;GALNT6;GALNT14;GALNT5;S...,hsa00512,0.361111
2,2,1293,N-Glycan biosynthesis,17/50,7.557622e-07,[Glycan biosynthesis and metabolism],KEGG_2021_Human,6.933598e-09,0.0,0.0,7.539137,141.636924,ST6GAL2;ALG5;ALG13;ALG3;MOGS;ALG10;FUT8;GANAB;...,hsa00510,0.340000
3,2,1293,Sphingolipid metabolism,15/49,1.411786e-05,[Lipid metabolism],KEGG_2021_Human,2.590433e-07,0.0,0.0,6.446078,97.762970,CERS3;CERS4;CERS6;CERK;SGMS1;SPHK2;SGPP2;NEU3;...,hsa00600,0.306122
4,2,1293,Glycosphingolipid biosynthesis,13/45,1.471350e-04,[],KEGG_2021_Human,3.374655e-06,0.0,0.0,5.927124,74.677125,B3GALNT1;ST8SIA1;B3GALT4;B3GALT5;FUT2;FUT4;B3G...,NaN,0.288889
5,2,1293,Glycosaminoglycan biosynthesis,13/53,8.703755e-04,[],KEGG_2021_Human,2.395529e-05,0.0,0.0,4.739668,50.426851,HS3ST3B1;GLCE;CSGALNACT2;EXTL2;FUT8;CHST11;DSE...,NaN,0.245283
6,2,1293,RNA transport,37/186,1.563500e-07,[],KEGG_2021_Human,7.172020e-10,0.0,0.0,3.669078,77.254880,NUP205;DDX20;SUMO4;NMD3;NXF1;EIF2B1;RPP14;RAE1...,NaN,0.198925
7,4,1158,Spliceosome,69/150,9.092484e-43,[Transcription],KEGG_2021_Human,3.852747e-45,0.0,0.0,14.675475,1500.824796,RBM25;EIF4A3;HNRNPU;PRPF19;PQBP1;EFTUD2;SNRPD2...,hsa03040,0.460000
8,4,1158,RNA polymerase,12/31,1.598606e-06,[Transcription],KEGG_2021_Human,6.773754e-08,0.0,0.0,10.373657,171.244436,POLR2B;POLR2C;POLR2D;POLR2E;POLR2F;POLR3G;POLR...,hsa03020,0.387097
9,4,1158,RNA degradation,24/79,4.959169e-10,"[Folding, sorting and degradation]",KEGG_2021_Human,8.405371e-12,0.0,0.0,7.229245,184.361288,HSPA9;BTG2;BTG1;PNPT1;ENO1;ENO2;LSM5;TOB1;LSM4...,hsa03018,0.303797


### Reactome

In [34]:
# Reactome enrichment
def reactome_enrichment(communities,
                        term_score_cap,
                        percentage):
    important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
    i = 0
    num_nonzero_communities = 0
    for community in communities:
        enr_path = gp.enrichr(
            gene_list=community,
            gene_sets=['Reactome_2022'],
            organism='Human',
            outdir=None
        )
        Reactome_df = enr_path.results

        # Filter by overlap percentage and adjusted p-value
        mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
        filtered = Reactome_df[mask].copy()
        
        # Categorization from Reactome Level 1
        filtered["Category"] = filtered["Term"].str.extract(r"(R-[A-Z]+-\d+)", expand=False).map(reactome_level1)
        
        # Sort
        filtered['Overlap (value)'] = filtered['Overlap'].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]))
        filtered = filtered.sort_values(['Overlap (value)'], ascending=False)
        
        # Add results to important terms
        if not filtered.empty:
            print(f"Size of community: {len(community)}")
            print(f"Number of filtered terms: {len(filtered)}")
            filtered.loc[:, "Community Index"] = i
            filtered.loc[:, "Community Size"] = len(community)
            important_terms = pd.concat([important_terms, filtered], ignore_index=True)
            display(HTML(filtered[["Community Index",'Term','Overlap','Adjusted P-value',"Category"]].head(30).to_html(max_cols=None)))
            num_nonzero_communities += 1
        i += 1
    print(f"{num_nonzero_communities} out of {len(communities)} communities had significant GO terms.")
    return important_terms

In [35]:
reactome_important_terms = reactome_enrichment(COMMUNITIES_HGNC,TERM_SCORE_CAP,PERCENTAGE)

Size of community: 1536
Number of filtered terms: 7


C:\Users\celem\AppData\Local\Temp\ipykernel_98744\3965740400.py:34: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  important_terms = pd.concat([important_terms, filtered], ignore_index=True)


,Community Index,Term,Overlap,Adjusted P-value,Category
4,0,rRNA Modification In Nucleus And Cytosol R-HSA-6790901,22/60,3.237469e-08,[Metabolism of RNA]
0,0,rRNA Processing R-HSA-72312,50/199,2.783598e-11,[Metabolism of RNA]
2,0,Major Pathway Of rRNA Processing In Nucleolus And Cytosol R-HSA-6791226,44/179,7.096132e-10,[Metabolism of RNA]
3,0,rRNA Processing In Nucleus And Cytosol R-HSA-8868773,45/189,9.879413e-10,[Metabolism of RNA]
6,0,CDC42 GTPase Cycle R-HSA-9013148,28/149,8.990148e-04,[Signal Transduction]
1,0,Metabolism Of RNA R-HSA-8953854,105/666,3.367127e-10,[Metabolism of RNA]
5,0,RHO GTPase Cycle R-HSA-9012999,62/441,3.259741e-04,[Signal Transduction]


Size of community: 1207
Number of filtered terms: 2


,Community Index,Term,Overlap,Adjusted P-value,Category
1,1,Formation Of Cornified Envelope R-HSA-6809371,21/74,1.202058e-07,[Developmental Biology]
0,1,Keratinization R-HSA-6805567,50/208,2.740079e-15,[Developmental Biology]


...

Size of community: 552
Number of filtered terms: 16


,Community Index,Term,Overlap,Adjusted P-value,Category
9,6,Lysosphingolipid And LPA Receptors R-HSA-419408,10/14,6.305485e-12,[Signal Transduction]
16,6,Relaxin Receptors R-HSA-444821,5/8,1.421974e-05,[Signal Transduction]
12,6,Nucleotide-like (Purinergic) Receptors R-HSA-418038,8/16,7.667705e-08,[Signal Transduction]
15,6,P2Y Receptors R-HSA-417957,6/12,6.340818e-06,[Signal Transduction]
4,6,Peptide Ligand-Binding Receptors R-HSA-375276,53/196,9.691555e-36,[Signal Transduction]
0,6,Class A/1 (Rhodopsin-like Receptors) R-HSA-373076,88/327,1.352930e-59,[Signal Transduction]
5,6,G Alpha (Q) Signaling Events R-HSA-416476,46/212,2.083612e-26,[Signal Transduction]
2,6,GPCR Ligand Binding R-HSA-500792,98/458,4.375962e-57,[Signal Transduction]
7,6,ADORA2B Mediated Anti-Inflammatory Cytokine Production R-HSA-9660821,26/131,7.805862e-14,[Disease]
14,6,Chemokine Receptors Bind Chemokines R-HSA-380108,11/56,6.071903e-06,[Signal Transduction]


Size of community: 275
Number of filtered terms: 3


,Community Index,Term,Overlap,Adjusted P-value,Category
2,8,Expression And Translocation Of Olfactory Receptors R-HSA-9752946,258/393,0.0,[Sensory Perception]
1,8,Olfactory Signaling Pathway R-HSA-381753,258/401,0.0,[Sensory Perception]
0,8,Sensory Perception R-HSA-9709957,258/616,0.0,[Sensory Perception]


7 out of 10 communities had significant GO terms.


In [36]:
reactome_important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,Overlap (value)
0,0,1536,rRNA Modification In Nucleus And Cytosol R-HSA...,22/60,3.237469e-08,[Metabolism of RNA],Reactome_2022,2.132720e-10,0.0,0.0,7.046027,156.904109,NOP56;WDR36;NOP14;KRR1;PNO1;UTP3;WDR3;FCF1;IMP...,0.366667
1,0,1536,rRNA Processing R-HSA-72312,50/199,2.783598e-11,[Metabolism of RNA],Reactome_2022,3.667455e-14,0.0,0.0,4.135917,127.951607,RBM28;RPL3;WDR3;RPLP0;FCF1;THUMPD1;WDR46;NOB1;...,0.251256
2,0,1536,Major Pathway Of rRNA Processing In Nucleolus ...,44/179,7.096132e-10,[Metabolism of RNA],Reactome_2022,2.804795e-12,0.0,0.0,4.003952,106.503882,RBM28;RPL3;WDR3;NIP7;RPL12;RPLP0;FCF1;ISG20L2;...,0.245810
3,0,1536,rRNA Processing In Nucleus And Cytosol R-HSA-8...,45/189,9.879413e-10,[Metabolism of RNA],Reactome_2022,5.206542e-12,0.0,0.0,3.839705,99.759777,RBM28;RPL3;WDR3;NIP7;RPL12;RPLP0;FCF1;ISG20L2;...,0.238095
4,0,1536,CDC42 GTPase Cycle R-HSA-9013148,28/149,8.990148e-04,[Signal Transduction],Reactome_2022,8.291309e-06,0.0,0.0,2.814762,32.933568,CPNE8;FAM13B;WIPF1;DOCK9;SNAP23;ARHGAP5;IQGAP3...,0.187919
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
204,6,552,Anti-inflammatory Response Favoring Leishmania...,26/165,1.705662e-11,[Disease],Reactome_2022,6.381727e-13,0.0,0.0,6.866455,192.811210,GPR27;GNAZ;GPR25;VIPR2;CALCB;GPR45;SCT;GPR20;G...,0.157576
205,6,552,Leishmania Infection R-HSA-9658195,26/247,1.197431e-07,[Disease],Reactome_2022,5.702052e-09,0.0,0.0,4.300380,81.631709,GPR27;GNAZ;GPR25;VIPR2;CALCB;GPR45;SCT;GPR20;G...,0.105263
206,8,275,Expression And Translocation Of Olfactory Rece...,258/393,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,2202.274510,inf,OR7G2;OR11H2;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;...,0.656489
207,8,275,Olfactory Signaling Pathway R-HSA-381753,258/401,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,2078.221308,inf,OR7G2;OR11H2;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;...,0.643392


### Disease Data Sets

In [37]:
# disease_term_score_cap = 0.001
# disease_percentage = 0.1
# important_diseases = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value"])

In [38]:
# # Disease-gene enrichment libraries
# disease_sets = [
#     'DisGeNET_2020', # curated gene–disease associations
#     'GWAS_Catalog_2023', # genome-wide association hits
#     'OMIM_Disease', # Mendelian disorders
#     'Jensen_DISEASES' # text-mined associations
# ]

# # # Disease-gene enrichment Analysis; save terms with small size and high p-value
# i = 0
# for community in communities_HGNC:
#     # Gene Ontology enrichment
#     enr_disease = gp.enrichr(
#         gene_list=community,
#         gene_sets=disease_sets,
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     enr_disease_df = enr_disease.results.sort_values('Adjusted P-value')
#     print(f"Size of community: {len(community)}")

#     mask =  (enr_disease_df["Adjusted P-value"] < disease_term_score_cap) & (enr_disease_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > disease_percentage))
        
#     filtered = enr_disease_df[mask].copy()
#     if not filtered.empty:
#         filtered.loc[:, "Community Index"] = i
#         filtered.loc[:, "Community Size"] = len(community)
#         important_diseases = pd.concat([important_diseases, filtered], ignore_index=True)

#     display(HTML(filtered[['Term','Overlap','Adjusted P-value']].head(10).to_html(max_cols=None)))
#     i += 1

# Important Terms df

In [39]:
important_terms = pd.DataFrame(columns=["Community Index","Community Size","Term", "Overlap", "Adjusted P-value","Category"])
c = [go_important_terms,kegg_important_terms,reactome_important_terms]
important_terms = pd.concat(c, ignore_index=True)
important_terms = important_terms.sort_values(by="Community Index")
important_terms

,Community Index,Community Size,Term,Overlap,Adjusted P-value,Category,Gene_set,P-value,Old P-value,Old Adjusted P-value,Odds Ratio,Combined Score,Genes,GO_ID,Slim_IDs,Overlap (value),KEGG_ID
0,0,1536,COPI Vesicle Coat (GO:0030126),7/12,6.302583e-04,[protein-containing complex],GO_Cellular_Component_2023,8.723299e-06,0.0,0.0,16.901635,1.968958e+02,COPA;COPB1;TMED3;COPZ2;COPG1;TMED7;COPE,GO:0030126,{GO:0032991},0.583333,NaN
909,0,1536,RHO GTPase Cycle R-HSA-9012999,62/441,3.259741e-04,[Signal Transduction],Reactome_2022,2.576871e-06,0.0,0.0,2.007121,2.582951e+01,ARHGAP11A;CPNE8;FAM13B;PLXND1;WIPF1;DOCK9;SOWA...,NaN,NaN,0.140590,NaN
908,0,1536,Metabolism Of RNA R-HSA-8953854,105/666,3.367127e-10,[Metabolism of RNA],Reactome_2022,8.872535e-13,0.0,0.0,2.341599,6.498090e+01,EIF4A2;EIF4A1;RPL3;GEMIN2;FCF1;HNRNPR;PHAX;THU...,NaN,NaN,0.157658,NaN
906,0,1536,rRNA Processing In Nucleus And Cytosol R-HSA-8...,45/189,9.879413e-10,[Metabolism of RNA],Reactome_2022,5.206542e-12,0.0,0.0,3.839705,9.975978e+01,RBM28;RPL3;WDR3;NIP7;RPL12;RPLP0;FCF1;ISG20L2;...,NaN,NaN,0.238095,NaN
905,0,1536,Major Pathway Of rRNA Processing In Nucleolus ...,44/179,7.096132e-10,[Metabolism of RNA],Reactome_2022,2.804795e-12,0.0,0.0,4.003952,1.065039e+02,RBM28;RPL3;WDR3;NIP7;RPL12;RPLP0;FCF1;ISG20L2;...,NaN,NaN,0.245810,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
865,8,275,Olfactory Receptor Activity (GO:0004984),220/362,0.000000e+00,[molecular transducer activity],GO_Molecular_Function_2023,0.000000e+00,0.0,0.0,551.633803,inf,OR7G2;OR52N1;OR2M5;OR2M4;OR10AC1;OR51L1;OR2AE1...,GO:0004984,{GO:0060089},0.607735,NaN
866,8,275,Sensory Perception Of Smell (GO:0007608),137/230,1.431229e-206,[multicellular organismal process],GO_Biological_Process_2023,8.418996e-208,0.0,0.0,209.567087,9.992310e+04,OR1C1;OR2M5;OR2M4;OR2T12;OR2T10;OR10AC1;OR2T11...,GO:0007608,{GO:0032501},0.595652,NaN
867,8,275,Sensory Perception Of Chemical Stimulus (GO:00...,61/110,2.837368e-85,[multicellular organismal process],GO_Biological_Process_2023,6.676161e-86,0.0,0.0,114.460805,2.244849e+04,OR10J1;OR1C1;OR10J5;OR8U9;OR8U8;OR2M4;OR8U3;OR...,GO:0007606,{GO:0032501},0.554545,NaN
1109,8,275,Expression And Translocation Of Olfactory Rece...,258/393,0.000000e+00,[Sensory Perception],Reactome_2022,0.000000e+00,0.0,0.0,2202.274510,inf,OR7G2;OR11H2;OR11H1;OR11H4;OR52N1;OR2M5;OR2M4;...,NaN,NaN,0.656489,NaN


In [40]:
important_terms.to_csv(f"../output/{DISEASE}/important_terms.csv", index=False)

# Robustness Analysis

In [ ]:
# def run_enrichment_func(community,term_score_cap,percentage):
#     # GO df
#     enr_go = gp.enrichr(
#         gene_list=community,
#         gene_sets=['GO_Biological_Process_2023',
#                 'GO_Molecular_Function_2023',
#                 'GO_Cellular_Component_2023'],
#         organism='Human',
#         outdir=None # don't write to disk
#     )
#     GO_df = enr_go.results
#     mask =  (GO_df["Adjusted P-value"] < term_score_cap) & (GO_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     GO_df = GO_df[mask].copy()   
    
#     # KEGG df
#     enr_kegg = gp.enrichr(
#         gene_list=community,
#         gene_sets=['KEGG_2021_Human'],
#         organism='Human',
#         outdir=None
#     )
#     KEGG_df = enr_kegg.results
#     mask =  (KEGG_df["Adjusted P-value"] < term_score_cap) & (KEGG_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     KEGG_df = KEGG_df[mask].copy() 
       
#     # Reactome df
#     enr_reactome = gp.enrichr(
#         gene_list=community,
#         gene_sets=['Reactome_2022'],
#         organism='Human',
#         outdir=None
#     )
#     Reactome_df = enr_reactome.results  
#     mask =  (Reactome_df["Adjusted P-value"] < term_score_cap) & (Reactome_df["Overlap"].apply(lambda x: int(x.split("/")[0])/int(x.split("/")[1]) > percentage))
#     Reactome_df = Reactome_df[mask].copy()
    
    
#     all_df = [GO_df,KEGG_df,Reactome_df]
#     # build result df by concatenating
#     result = pd.concat(all_df, ignore_index=True)
#     return result

In [ ]:
# from json import JSONDecodeError

# # ---------------- 1) Safe wrapper that calls YOUR enrichr function ----------------
# _ENR_CACHE = {}  # key: tuple(sorted(genes)) -> DataFrame (copy)

# def run_enrichment_safe(run_enrichment_func, community, retries=5, base_sleep=0.8):
#     """
#     Calls user's run_enrichment_func(community) with retries + memoization.
#     Returns a DataFrame (possibly empty). Never raises JSONDecodeError outward.
#     """
#     # Ensure we always pass a list of gene symbols (never a bare string)
#     genes = np.atleast_1d(np.array(community, dtype=object)).tolist()
#     if len(genes) == 0:
#         return pd.DataFrame()

#     key = tuple(sorted(genes))
#     if key in _ENR_CACHE:
#         return _ENR_CACHE[key].copy()

#     for a in range(retries):
#         try:
#             df = run_enrichment_func(genes,TERM_SCORE_CAP,PERCENTAGE)
#             if df is None:
#                 # treat as transient failure to trigger retry
#                 raise RuntimeError("run_enrichment_func returned None")
#             _ENR_CACHE[key] = df.copy()
#             return df
#         except (JSONDecodeError, OSError, RuntimeError, ValueError) as e:
#             # Transient errors from HTTP/JSON/file handling inside gseapy
#             if a == retries - 1:
#                 # Give up: return empty so pipeline continues
#                 return pd.DataFrame()
#             time.sleep(base_sleep * (2 ** a) + np.random.rand() * 0.3)

#     return pd.DataFrame()

# # ---------------- 2) Minimal bootstrap to record robust terms ----------------
# def get_robust_terms(communities_HGNC, run_enrichment_func,
#                      R=50, leaveout=0.10, recurrence_cutoff=0.70, seed=42):
#     """
#     Uses YOUR run_enrichment_func(community)->DataFrame (already filtered to significant terms).
#     Returns DataFrame with columns: community_id, term, recurrence (and Gene_set if available).
#     """
#     rng = np.random.default_rng(seed)
#     rows = []

#     for cid, community in enumerate(communities_HGNC):
#         n = len(community)
#         if n == 0:
#             continue
#         drop_k = max(1, int(np.floor(leaveout * n)))
#         counts = Counter()

#         for _ in range(R):
#             # Jackknife subset (ensure not empty)
#             keep = np.ones(n, dtype=bool)
#             keep[rng.choice(n, size=min(drop_k, n), replace=False)] = False
#             sub = np.atleast_1d(np.array(community, dtype=object)[keep]).tolist()
#             if len(sub) == 0:
#                 continue

#             df = run_enrichment_safe(run_enrichment_func, sub)
#             if df is None or df.empty:
#                 continue

#             # Your function already returns significant terms; just count them.
#             # If it includes multiple libraries, preserve Gene_set to disambiguate names.
#             if 'Term' not in df.columns:
#                 continue  # be defensive

#             if 'Gene_set' in df.columns:
#                 terms = (df[['Term', 'Gene_set']]
#                          .dropna()
#                          .drop_duplicates()
#                          .apply(lambda r: f"{r['Term']}|{r['Gene_set']}", axis=1)
#                          .tolist())
#             else:
#                 terms = df['Term'].dropna().drop_duplicates().tolist()

#             counts.update(terms)

#             # tiny pause helps with API rate limits if your func calls Enrichr internally
#             time.sleep(0.03)

#         # Keep only robust terms
#         for t, c in counts.items():
#             freq = c / max(R, 1)
#             if freq >= recurrence_cutoff:
#                 if '|' in t:
#                     term, gene_set = t.split('|', 1)
#                     rows.append({'Community Index': cid, 'Term': term, 'recurrence': freq, 'Gene_set': gene_set})
#                 else:
#                     rows.append({'Community Index': cid, 'Term': t, 'recurrence': freq})

#     return (pd.DataFrame(rows)
#               .sort_values(['Community Index', 'recurrence'], ascending=[True, False])
#               .reset_index(drop=True))

In [ ]:
# twr3 = get_robust_terms([COMMUNITIES_HGNC[1]], run_enrichment_func,
#                                 R=25, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
# twr3

In [ ]:
# terms_with_recurrence = get_robust_terms(COMMUNITIES_HGNC, run_enrichment_func,
#                                 R=10, leaveout=0.1, recurrence_cutoff=0)

In [ ]:
# terms_with_recurrence

In [ ]:
# # rename important terms to match terms_with_recurrence
# important_terms = important_terms.rename(columns={'index': 'community_id'})
# important_terms = important_terms.rename(columns={'Term': 'term'})

In [ ]:
# terms_with_rec_merged = important_terms.merge(
#     terms_with_recurrence[['community_id', 'term', 'Gene_set', 'recurrence']],
#     on=['community_id', 'term', 'Gene_set'],
#     how='left'
# )

# terms_with_rec_merged['recurrence'] = terms_with_rec_merged['recurrence'].fillna(0.0)

# terms_with_rec_merged = terms_with_rec_merged.sort_values(
#     ['community_id', 'recurrence'],
#     ascending=[True, False]
# ).reset_index(drop=True)

In [ ]:
# terms_with_rec_merged

In [ ]:
# community_summary = (
#     terms_with_rec_merged
#     .groupby("community_id")["recurrence"]
#     .agg(mean_recurrence="mean", term_count="count")
#     .reset_index()
# )

# print(community_summary)

In [ ]:
# display(HTML(terms_with_recurrence.to_html(max_cols=None)))

# Checks!

In [ ]:
DGIDB_genes_ncbi = list(DGIDB_gene_to_index.keys())

In [ ]:
def DGIDB_count(c):
    return len(set(c) & set(DGIDB_genes_ncbi))